# 04 · Feature groups: check and summary

**Tracker task:** IMP-05 · Divide variables into clinical, hormonal/biochemical, metabolic and morphology groups

**Evidence required:** a table mapping every variable to one group.

The grouping itself is decided **by hand** in `docs/feature_groups.xlsx` (sheet `feature_groups`).
This notebook does not decide anything. It:
1. checks that the spreadsheet matches the dataset exactly and is complete, and
2. summarises the agreed grouping (the IMP-05 evidence).


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"
GROUPS_FILE = ROOT / "docs" / "feature_groups.xlsx"
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 80)

# The dataset's features, named exactly as in notebooks 01-03
df = pd.read_excel(DATA_FILE, sheet_name="Full_new")
df.columns = df.columns.str.strip()
FEATURES = [c for c in df.columns
            if c not in ["Sl. No", "Patient File No.", "PCOS (Y/N)"] and not c.startswith("Unnamed")]

# Your grouping (header is on row 4 of the sheet)
groups = pd.read_excel(GROUPS_FILE, sheet_name="feature_groups", header=3)
groups = groups[["variable", "group", "acquisition", "variable_type", "reason"]].dropna(how="all")
for col in groups.columns:
    groups[col] = groups[col].astype("string").str.strip()     # ignore stray spaces typed into cells

ALLOWED = {
    "group": {"clinical", "hormonal_biochemical", "metabolic", "morphology", "excluded"},
    "acquisition": {"self_report", "physical_exam", "blood_test", "ultrasound"},
    "variable_type": {"binary", "nominal", "count", "continuous"},
}
print(f"Dataset features: {len(FEATURES)} | rows in spreadsheet: {len(groups)}")

Dataset features: 41 | rows in spreadsheet: 41


## Check 1 · Do the names match the dataset exactly?

In [2]:
in_sheet = groups["variable"].dropna().tolist()
not_in_sheet = sorted(set(FEATURES) - set(in_sheet))          # dataset variables you have not listed
unknown = sorted(set(in_sheet) - set(FEATURES))               # names in the sheet that are not in the dataset
duplicated = sorted(groups.loc[groups["variable"].duplicated(), "variable"])

name_issues = (
    [("Missing from spreadsheet", v) for v in not_in_sheet]
    + [("Not a dataset variable (check spelling)", v) for v in unknown]
    + [("Listed more than once", v) for v in duplicated]
)
pd.DataFrame(name_issues, columns=["issue", "variable"]) if name_issues else print("All 41 names match, each listed once.")

All 41 names match, each listed once.


## Check 2 · Is every row filled in with allowed values?

In [3]:
fill_issues = []
for _, row in groups.iterrows():
    for col, allowed in ALLOWED.items():
        value = row[col]
        if pd.isna(value):
            fill_issues.append((row["variable"], col, "blank"))
        elif value not in allowed:
            fill_issues.append((row["variable"], col, f"'{value}' is not an allowed value"))
    if row["group"] == "excluded" and pd.isna(row["reason"]):
        fill_issues.append((row["variable"], "reason", "required for excluded variables"))

if fill_issues:
    issues = pd.DataFrame(fill_issues, columns=["variable", "column", "issue"])
    print(f"{issues['variable'].nunique()} variable(s) not yet complete:")
    display(issues.groupby(["column", "issue"]).size().to_frame("rows"))
else:
    print("Every row is complete and uses allowed values.")

Every row is complete and uses allowed values.


## Result

In [4]:
READY = not name_issues and not fill_issues
if READY:
    print("PASSED: the feature-group dictionary is complete and matches the dataset.")
else:
    print(f"NOT READY: {len(name_issues)} name issue(s) and {len(fill_issues)} filling issue(s). "
          "Fix them in docs/feature_groups.xlsx and run this notebook again.")

PASSED: the feature-group dictionary is complete and matches the dataset.


## Summary · Variables per group (IMP-05 evidence)

Meaningful once the check above has passed.

In [5]:
summary = (groups.fillna({"group": "(blank)"})
           .groupby("group")["variable"]
           .agg(variables="count", members=lambda s: ", ".join(s)))
summary

,variables,members
group,,
clinical,15,"Age (yrs), Pulse rate(bpm), RR (breaths/min), Cycle(R/I), Cycle length(days)..."
excluded,3,"Blood Group, I beta-HCG(mIU/mL), II beta-HCG(mIU/mL)"
hormonal_biochemical,9,"Hb(g/dl), FSH(mIU/mL), LH(mIU/mL), FSH/LH, TSH (mIU/L), AMH(ng/mL), PRL(ng/m..."
metabolic,9,"Weight (Kg), Height(Cm), BMI, Hip(inch), Waist(inch), Waist:Hip Ratio, RBS(m..."
morphology,5,"Follicle No. (L), Follicle No. (R), Avg. F size (L) (mm), Avg. F size (R) (m..."


## Group versus acquisition source

Where the physiological group and the way a value is obtained differ (for example a metabolic marker measured by a
blood test), a missing-information scenario such as *no laboratory results* removes variables from more than one group.

In [6]:
pd.crosstab(groups["group"].fillna("(blank)"), groups["acquisition"].fillna("(blank)"),
            margins=True, margins_name="total")

acquisition,blood_test,physical_exam,self_report,ultrasound,total
group,,,,,
clinical,0,2,13,0,15
excluded,3,0,0,0,3
hormonal_biochemical,9,0,0,0,9
metabolic,1,8,0,0,9
morphology,0,0,0,5,5
total,13,10,13,5,41


## Variable types and exclusions

In [7]:
display(groups["variable_type"].fillna("(blank)").value_counts().to_frame("variables"))

excluded = groups[groups["group"] == "excluded"][["variable", "reason"]]
excluded if len(excluded) else print("No variables excluded.")

,variables
variable_type,
continuous,27
binary,9
count,4
nominal,1


,variable,reason
4,Blood Group,No established link to PCOS; nominal codes 11-18 would be misread as a numer...
13,I beta-HCG(mIU/mL),"Pregnancy hormone, not a PCOS marker; extremely skewed and its class differe..."
14,II beta-HCG(mIU/mL),"Pregnancy hormone; copied from beta-HCG I in 57% of records (IMP-03), so lar..."


## Next steps

- Agree the grouping with supervisor, especially the ambiguous variables (e.g. skin darkening, weight gain,
  height, blood pressure, endometrium, beta-HCG) and the exclusions.
- Record any change in `docs/feature_groups.xlsx` and rerun this notebook until it shows **PASSED**.